In [ ]:
import pandas as pd
from pandas.errors import PerformanceWarning
import warnings

warnings.simplefilter('ignore', category=PerformanceWarning)

In [ ]:
from src.features.load_day_ahead_dataset import load_day_ahead_dataset

start = pd.Timestamp("2024-07-01")
end = pd.Timestamp("2025-04-01")
val_cutoff = pd.Timestamp("2025-01-01")
select_cutoff = pd.Timestamp("2025-02-01")
test_cutoff = pd.Timestamp("2025-03-01")

df = load_day_ahead_dataset(start, end)

In [ ]:
from src.models.data_split import time_series_split, DA_KNOWN
from src.models.train_day_ahead_time_series import train_day_ahead_time_series

ts_train, ts_val, ts_select, ts_test = time_series_split(df, val_cutoff, select_cutoff, test_cutoff)
ts_metrics = ["MAE", "RMSE"]

ts_mae_model = train_day_ahead_time_series(ts_train, ts_val, "MAE")
display(ts_mae_model.leaderboard(ts_select, extra_metrics=ts_metrics))

ts_rmse_model = train_day_ahead_time_series(ts_train, ts_val, "RMSE")
display(ts_rmse_model.leaderboard(ts_select, extra_metrics=ts_metrics))

In [ ]:
from src.models.data_split import tabular_split
from src.models.train_day_ahead_tabular import train_day_ahead_tabular

tab_df = df[[col for col in df.columns if any(col.startswith(prefix) for prefix in DA_KNOWN)] + ["DA Price (EUR/MWh)"]]
tab_train, tab_val, tab_select, tab_test = tabular_split(tab_df, val_cutoff, select_cutoff, test_cutoff)
tab_metrics = ["mae", "rmse"]

tab_mae_model = train_day_ahead_tabular(tab_train, tab_val, "mae")
display(tab_mae_model.leaderboard(tab_test, extra_metrics=tab_metrics))

tab_rmse_model = train_day_ahead_tabular(tab_train, tab_val, "rmse")
display(tab_rmse_model.leaderboard(tab_test, extra_metrics=tab_metrics))